# 🎬 FilmDB — 3-Stage RAG Recommendation Engine
> **A production-grade movie recommendation system built with BGE embeddings, a cross-encoder reranker, and LLaMA-3.3-70B curation.**

| Component | Technology |
|---|---|
| **Corpus** | 95,090 curated movies |
| **Bi-Encoder** | `BAAI/bge-base-en-v1.5` (768-d) |
| **Reranker** | `BAAI/bge-reranker-large` (raw logits) |
| **LLM** | `llama-3.3-70b-versatile` via Groq API |
| **Vector DB** | Qdrant Cloud |
| **Evaluation** | 21 queries across 9 retrieval strategies |

> ⚡ **This notebook runs on CPU only — no GPU or API key needed.**
> All results below are pre-computed from a live T4 GPU run and stored in `showcase_results.json`.
> To interact with the live recommendation engine, visit the [FilmDB web app](https://your-filmdb-url.com).

---

## 1. Pipeline Architecture
The recommendation engine is a **3-stage pipeline** designed to progressively refine candidates:

In [1]:
import base64
from IPython.display import Image, display, HTML, Markdown

# Note: Mermaid diagrams are rendered via mermaid.ink (requires internet)
def render_mermaid(graph_code):
    b64 = base64.urlsafe_b64encode(graph_code.encode('utf8')).decode('ascii')
    display(HTML(f'<img src="https://mermaid.ink/img/{b64}" style="width:100%; max-width:800px;">'))

render_mermaid('''
graph TD
    U(["🔍 User Query"]) --> A
    subgraph Stage1 ["Stage 1 — Dense Retrieval"]
        A["BGE-base Bi-Encoder\n768-d Embeddings"] --> B["Cosine Similarity\nTop-50 Candidates"]
    end
    subgraph Stage2 ["Stage 2 — Cross-Encoder Rerank"]
        B --> C["BGE-reranker-large\nRaw Logit Scoring"]
        C --> D["Top-10 Refined"]
    end
    subgraph Stage3 ["Stage 3 — LLM Curation"]
        D --> E["llama-3.3-70b-versatile\nvia Groq API"]
        E --> F(["🎬 Top-5 + Explanation"])
    end
    style Stage1 fill:#1a1a2e,color:#fff
    style Stage2 fill:#16213e,color:#fff
    style Stage3 fill:#0f3460,color:#fff
''')

## 2. Retrieval Strategy Catalog
FilmDB uses **9 distinct retrieval strategies**, each with different filter logic:

In [2]:
import pandas as pd

strategies = pd.DataFrame([
    {'Strategy': 'S1_Narrative',  'Description': 'Pure semantic plot search',      'Filter': 'None',                    'Example': 'mind-bending movie about dreams'},
    {'Strategy': 'S2_Prestige',   'Description': 'Oscar-winning films only',       'Filter': 'oscar_wins ≥ 1',          'Example': 'sweeping historical epic'},
    {'Strategy': 'S3_Cultural',   'Description': 'Language / regional cinema',     'Filter': 'original_language = X',   'Example': 'Japanese revenge thriller'},
    {'Strategy': 'S4_Platform',   'Description': 'Streaming service filter',       'Filter': 'netflix = 1 (etc.)',      'Example': 'action movie on Netflix'},
    {'Strategy': 'S5_Similarity', 'Description': 'Semantic similarity to a film',  'Filter': 'None',                    'Example': 'movies similar to The Matrix'},
    {'Strategy': 'S6_Director',   'Description': 'Director spotlight / best-of',   'Filter': 'director = X',            'Example': 'best Stanley Kubrick films'},
    {'Strategy': 'S7_Diversity',  'Description': 'Diverse eras & sub-genres',      'Filter': 'Decade bucketing',        'Example': 'diverse list of thrillers'},
    {'Strategy': 'S8_Mood',       'Description': 'Emotional vibe matching',        'Filter': 'None',                    'Example': 'cozy rainy-day movie'},
    {'Strategy': 'S9_HiddenGem',  'Description': 'Obscure high-quality films',     'Filter': 'votes<10k, rating≥7.0',   'Example': 'masterpiece no one talks about'},
])

display(strategies.style
    .set_table_styles([{'selector': 'th', 'props': [('background', '#0f3460'), ('color', 'white'), ('font-size', '13px')]}])
    .hide(axis='index')
)

Strategy,Description,Filter,Example
S1_Narrative,Pure semantic plot search,None,mind-bending movie about dreams
S2_Prestige,Oscar-winning films only,oscar_wins ≥ 1,sweeping historical epic
S3_Cultural,Language / regional cinema,original_language = X,Japanese revenge thriller
S4_Platform,Streaming service filter,netflix = 1 (etc.),action movie on Netflix
S5_Similarity,Semantic similarity to a film,None,movies similar to The Matrix
S6_Director,Director spotlight / best-of,director = X,best Stanley Kubrick films
S7_Diversity,Diverse eras & sub-genres,Decade bucketing,diverse list of thrillers
S8_Mood,Emotional vibe matching,None,cozy rainy-day movie
S9_HiddenGem,Obscure high-quality films,"votes<10k, rating≥7.0",masterpiece no one talks about


## 3. Load Precomputed Results
Results from all 21 evaluation queries, pre-run on a Kaggle T4 GPU:

In [3]:
import json, os
from pathlib import Path

# Dynamically search for the dataset (handles any Kaggle dataset name)
json_paths = list(Path('/kaggle/input').rglob('showcase_results.json'))
json_paths.append(Path('showcase_results.json')) # Local fallback

results = None
for p in json_paths:
    if p.exists():
        with open(p, 'r', encoding='utf-8') as f:
            results = json.load(f)
        print(f'Loaded {len(results)} queries from {p}')
        break

if results is None:
    raise FileNotFoundError('showcase_results.json not found. Please upload it as a Kaggle dataset.')


Loaded 21 queries from /kaggle/input/datasets/naren2308/filmdb-recommendation-engine-showcase/showcase_results.json


## 4. Interactive Query Explorer
> **💡 Live interactivity (dropdowns) only works when you are actively running this notebook in a Kaggle session.**
> If you are viewing this as a static output, scroll down to **Section 5** to see the static Plotly charts and **Section 6** for curated highlights.


In [4]:
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

STRATEGY_COLORS = {
    'S1_Narrative':  '#e76f51', 'S2_Prestige':  '#2a9d8f', 'S3_Cultural':  '#e9c46a',
    'S4_Platform':   '#264653', 'S5_Similarity':'#118ab2', 'S6_Director':  '#d62828',
    'S7_Diversity':  '#8338ec', 'S8_Mood':      '#06d6a0', 'S9_HiddenGem': '#fb8500',
}

def fmt_table(rows, score_label):
    html = f'<table style="width:100%;border-collapse:collapse;font-size:13px">'
    html += f'<tr style="background:#0f3460;color:white"><th>Title</th><th>Year</th><th>IMDB</th><th>Votes</th><th>{score_label}</th></tr>'
    for i, r in enumerate(rows):
        bg = '#1a1a2e' if i % 2 == 0 else '#16213e'
        html += f'<tr style="background:{bg};color:#eee">'
        html += f'<td>{r["title"]}</td><td>{r["year"]}</td><td>{r["rating"]}</td><td>{r["votes"]:,}</td><td>{r["score"]:.4f}</td>'
        html += '</tr>'
    html += '</table>'
    return html

query_dropdown = widgets.Dropdown(
    options=[(f"{r['id']}: {r['query'][:75]}", i) for i, r in enumerate(results)],
    description='Query:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='90%'),
)

output = widgets.Output()

def show_query(change):
    with output:
        clear_output(wait=True)
        r = results[change['new']]
        color = STRATEGY_COLORS.get(r['strategy'], '#888')
        display(HTML(f'<h3 style="color:{color}">🔍 {r["query"]}</h3>'))
        display(HTML(f'<p>Strategy: <code>{r["strategy"]}</code> | Time: <b>{r["elapsed"]}s</b></p>'))
        display(HTML('<h4>Stage 1 — Dense Retrieval (Cosine Similarity)</h4>'))
        display(HTML(fmt_table(r['mode_a'], 'Sim Score')))
        display(HTML('<h4>Stage 2 — Cross-Encoder Rerank</h4>'))
        display(HTML(fmt_table(r['mode_b'], 'Rerank Score')))
        display(HTML('<h4>Stage 3 — LLM Curation (llama-3.3-70b-versatile)</h4>'))
        display(Markdown(r["mode_c"]))

query_dropdown.observe(show_query, names='value')
display(query_dropdown, output)
# Trigger once for initial display
show_query({'new': 0})


Dropdown(description='Query:', layout=Layout(width='90%'), options=(('Q01: Recommend me a mind-bending movie a…

Output()

## 5. Pipeline Analysis (Plotly)
> These charts are **fully interactive** — hover, zoom, and click even in static view.

In [5]:
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio

# Use iframe renderer to ensure charts display in Kaggle's static view
pio.renderers.default = 'iframe'

DARK = '#0f3460'
TEMPLATE = 'plotly_dark'

# Chart 1: Average IMDB Rating — Mode A vs Mode B (reranking quality lift)
rows = []
for r in results:
    a_avg = sum(x['rating'] or 0 for x in r['mode_a']) / len(r['mode_a'])
    b_avg = sum(x['rating'] or 0 for x in r['mode_b']) / len(r['mode_b'])
    rows.append({'Query': r['id'], 'Dense Only': round(a_avg, 2), 'After Rerank': round(b_avg, 2)})

df_lift = pd.DataFrame(rows).melt(id_vars='Query', var_name='Stage', value_name='Avg IMDB Rating')
fig1 = px.bar(df_lift, x='Query', y='Avg IMDB Rating', color='Stage', barmode='group',
    title='📊 Reranker Quality Lift: Avg IMDB Rating Before vs After Reranking',
    color_discrete_map={'Dense Only': '#e76f51', 'After Rerank': '#2a9d8f'},
    template=TEMPLATE, height=420)
fig1.update_layout(legend=dict(orientation='h', y=1.05))
fig1.show()

# Chart 2: Query response time by strategy
df_time = pd.DataFrame([{'Strategy': r['strategy'], 'Time (s)': r['elapsed'], 'Query': r['id']} for r in results])
fig2 = px.strip(df_time, x='Strategy', y='Time (s)', color='Strategy', hover_data=['Query'],
    title='⏱ Query Latency by Strategy (3-stage pipeline)',
    template=TEMPLATE, height=380)
fig2.show()

# Chart 3: Score spread — reranker discriminative power
score_rows = []
for r in results:
    for rec in r['mode_b']:
        score_rows.append({'Query': r['id'], 'Rerank Score': rec['score'], 'Strategy': r['strategy']})

df_scores = pd.DataFrame(score_rows)
fig3 = px.box(df_scores, x='Query', y='Rerank Score', color='Strategy',
    title='🎯 Cross-Encoder Score Distribution per Query (raw logits)',
    template=TEMPLATE, height=420)
fig3.update_layout(showlegend=False)
fig3.show()


## 6. Head-to-Head Highlights
These 4 queries best demonstrate the pipeline's power — **notice how each stage improves the results:**

In [6]:
HIGHLIGHTS = ['Q02', 'Q15', 'Q06', 'Q08']
TITLES = {
    'Q02': '🌸 Slice of Life — Finding quiet Tokyo family dramas',
    'Q15': '🎬 Director Spotlight — The essential Stanley Kubrick roadmap',
    'Q06': '🎷 Oscar-Winning Jazz Musical — Reranker surfaces Whiplash & La La Land',
    'Q08': '🔫 Indian Gangster Epic — Cross-language retrieval spanning all Indian cinemas',
}

for qid in HIGHLIGHTS:
    r = next(x for x in results if x['id'] == qid)
    display(HTML(f'<hr><h3>{TITLES[qid]}</h3>'))
    display(HTML(f'<p><em>{r["query"]}</em></p>'))
    cols = '<div style="display:grid;grid-template-columns:1fr 1fr;gap:12px">'
    cols += f'<div><b>Stage 1 (Dense)</b>{fmt_table(r["mode_a"][:5], "Sim")}</div>'
    cols += f'<div><b>Stage 2 (Reranked)</b>{fmt_table(r["mode_b"][:5], "Score")}</div>'
    cols += '</div>'
    display(HTML(cols))
    display(HTML(f'<p><b>Stage 3 — LLM Curation:</b></p>'))
    display(Markdown(r["mode_c"]))


Title,Year,IMDB,Votes,Sim
Tokyo Story,1953,8.1,"65,916",0.6806
Tokyo Family,2013,7.5,"2,357",0.6743
Still Walking,2008,7.9,"17,811",0.6249
The Taste of Tea,2004,7.6,"7,835",0.6245
An Inn in Tokyo,1935,7.4,"1,820",0.6186
Title,Year,IMDB,Votes,Score
Tokyo Family,2013,7.5,"2,357",0.2487
Tokyo Sonata,2008,7.5,"11,285",0.2302
My Neighbors the Yamadas,1999,7.1,"15,313",0.1142
Kabei: Our Mother,2008,7.6,"1,583",0.0752


### Recommended Films
| Film | Why it fits | Caveats |
| --- | --- | --- |
| **Tokyo Family (2013)** | This film fits the request as it is a quiet, slice-of-life story about a family in Tokyo, exploring themes of tradition, change, and family bonds. Directed by Yoji Yamada, it offers a nuanced portrayal of Japanese family life. The film's pace and focus on character development make it an ideal choice. | Japanese language, subtle humor |
| **Tokyo Sonata (2008)** | Tokyo Sonata is a drama that explores the lives of a typical Tokyo family, delving into themes of identity, connection, and the search for meaning in a busy city. Kiyoshi Kurosawa's direction brings a unique, often surreal perspective to the traditional family drama. It's a thought-provoking film that captures the essence of family life in Tokyo. | Some scenes may be interpreted as surreal or dreamlike, Japanese language |
| **My Neighbors the Yamadas (1999)** | This animated film, directed by Isao Takahata, offers a comedic and heartwarming look at the daily life of the Yamada family in Tokyo. It's a slice-of-life story that explores the quirks and charms of family life, making it a great fit for the request. The animation style and humor add a unique touch to the traditional family drama. | Animated, Japanese language, comedic tone |
| **The Taste of Tea (2004)** | The Taste of Tea is a comedy-drama that follows the lives of the Haruno family in the countryside just outside of Tokyo. While not exclusively set in Tokyo, it captures the essence of family life in Japan, exploring themes of identity, love, and the complexities of family relationships. The film's blend of comedy and fantasy elements adds a unique layer to its portrayal of family life. | Some fantasy elements, Japanese language, rural setting near Tokyo |
| **Nobody Knows (2004)** | Directed by Hirokazu Koreeda, Nobody Knows tells the story of four siblings left to fend for themselves in a Tokyo apartment. The film is a powerful and poignant exploration of family, love, and resilience. It fits the request by offering a deep, quiet look at the lives of a family in Tokyo, albeit under challenging circumstances. | Japanese language, themes of abandonment and poverty, highly acclaimed but emotionally demanding |

Title,Year,IMDB,Votes,Sim
2001: A Space Odyssey,1968,8.3,"699,561",0.5577
Dr. Strangelove or: How I Learned to Stop Worrying and Love the Bomb,1964,8.4,"507,326",0.5485
Spartacus,1960,7.9,"140,269",0.5470
Lolita,1962,7.5,"106,453",0.5446
A Clockwork Orange,1971,8.3,"861,107",0.5440
Title,Year,IMDB,Votes,Score
Barry Lyndon,1975,8.1,"177,385",0.9170
Dr. Strangelove or: How I Learned to Stop Worrying and Love the Bomb,1964,8.4,"507,326",0.4187
2001: A Space Odyssey,1968,8.3,"699,561",0.4165
A Clockwork Orange,1971,8.3,"861,107",0.3767


### Stanley Kubrick Film Recommendations
The following table lists the top 5 Stanley Kubrick films that satisfy the request, along with a brief explanation of why each film fits and any relevant caveats.

| Film | Why it fits | Caveats |
| --- | --- | --- |
| **2001: A Space Odyssey (1968)** | This seminal science fiction film is a masterpiece of visual storytelling, exploring themes of human evolution, technology, and existentialism. Its influence on the genre is unparalleled, and it remains a must-see for cinephiles. With its slow-burning pace and enigmatic ending, it's a film that rewards multiple viewings. | Slow pace, abstract themes may challenge some viewers. |
| **A Clockwork Orange (1971)** | This dystopian satire is a powerful commentary on free will, violence, and societal conditioning. Kubrick's innovative direction and the film's bold visuals make it a standout in his oeuvre. However, its graphic content and dark themes may not be suitable for all audiences. | Graphic violence, nudity, and mature themes; not suitable for all ages. |
| **Dr. Strangelove or: How I Learned to Stop Worrying and Love the Bomb (1964)** | This black comedy is a scathing critique of Cold War politics and the military-industrial complex. Kubrick's masterful balance of humor and satire makes it a compelling watch, with a talented ensemble cast bringing the absurd story to life. | Some viewers may find the film's tone and themes dated or off-putting. |
| **Barry Lyndon (1975)** | This period drama is a visually stunning adaptation of William Makepeace Thackeray's novel, exploring themes of class, morality, and the human condition. Kubrick's meticulous attention to detail and innovative cinematography make it a work of art. | Slow pace and period setting may not appeal to all viewers; some scenes depict violence and mature themes. |
| **Full Metal Jacket (1987)** | This war drama is a intense and thought-provoking exploration of the psychological effects of military training and combat. Kubrick's direction is characteristically meticulous, and the film's performances are outstanding. However, its graphic content and mature themes may not be suitable for all audiences. | Graphic violence, strong language, and mature themes; not suitable for all ages. |

These five films represent some of the best and most essential works in Stanley Kubrick's oeuvre, showcasing his innovative direction, visual style, and thought-provoking themes.

Title,Year,IMDB,Votes,Sim
All That Jazz,1979,7.8,"34,088",0.6302
Shine,1996,7.6,"55,823",0.6238
King of Jazz,1930,6.7,"1,876",0.6208
Oklahoma!,1955,7.0,"13,596",0.6052
Tango,1998,7.0,"3,259",0.6038
Title,Year,IMDB,Votes,Score
La La Land,2016,8.0,"636,238",0.3308
Whiplash,2014,8.5,"934,655",0.3191
Soul,2020,8.0,"358,317",0.1948
Alexander's Ragtime Band,1938,6.8,"2,485",0.1548


### Recommended Films
| Film | Why it fits | Caveats |
| --- | --- | --- |
| **La La Land (2016)** | La La Land is a modern classic that beautifully captures the essence of jazz and passion through its stunning musical numbers and romantic storyline. It's an Oscar-winning film that pays homage to the golden age of Hollywood musicals while maintaining a contemporary feel. With its vibrant colors and memorable performances, it perfectly fits the request. | None |
| **Whiplash (2014)** | Whiplash is an intense drama about jazz that explores the passion and dedication required to excel in the competitive world of music. Although not a traditional musical, it features powerful jazz performances and delves into the psychological aspects of ambition and perfection. Directed by Damien Chazelle, it's a critically acclaimed film that won several Oscars. | Intense scenes, strong language |
| **All That Jazz (1979)** | All That Jazz is a semi-autobiographical musical drama by Bob Fosse that combines jazz, passion, and the highs and lows of a show business career. The film is known for its stunning choreography, memorable songs, and a deep exploration of the protagonist's life. It's a classic that fits the request with its blend of music, drama, and the allure of jazz. | Mature themes, some nudity |
| **'Round Midnight (1986)** | 'Round Midnight is a drama about jazz that tells the story of an American jazz musician in Paris, exploring themes of passion, addiction, and the power of music to transcend borders and cultures. The film features outstanding jazz performances and is a poignant tribute to the art form. It won an Oscar for its original score, making it a fitting choice. | Some mature themes, smoking |
| **Soul (2020)** | Soul is an animated film that, while not exclusively about jazz, features it prominently as a central theme. The movie follows a music teacher who dreams of becoming a jazz performer and, through a series of events, learns valuable lessons about passion, purpose, and the soul of music. It's an Oscar-winning film that beautifully captures the essence of jazz and its ability to inspire and uplift. | Suitable for all ages, but deals with existential themes |

Title,Year,IMDB,Votes,Sim
Sitapur: The City of Gangsters,2021,5.6,29,0.6457
Gangster,2014,3.4,568,0.6456
Temper,2015,7.4,"8,720",0.6357
Mufti,2017,7.8,"3,210",0.6266
Mafia,1996,5.0,83,0.6249
Title,Year,IMDB,Votes,Score
Nayakan,1987,8.6,"21,576",0.0248
Vada Chennai,2018,8.4,"18,773",0.0181
Psycho Raman,2016,7.3,"15,615",0.0110
Insaniyat,1994,3.0,352,0.0104


### Recommended Films
| Film | Why it fits | Caveats |
| --- | --- | --- |
| **Nayakan (1987)** | This film fits the request as it is a seminal work in Indian cinema that explores the life of a gangster, delving into the criminal underworld, police encounters, and the consequences of street violence. Directed by Mani Ratnam, it offers a gritty and realistic portrayal. With its high rating, it's a classic choice for those interested in the genre. | The film is in Tamil, so subtitles may be necessary for non-Tamil speakers. |
| **Vada Chennai (2018)** | Vada Chennai is a more recent epic that explores the criminal underworld of North Chennai, focusing on gang wars, police brutality, and the struggle for power. It's known for its raw and intense depiction of street life. The film's high rating and contemporary setting make it a compelling choice. | Like Nayakan, Vada Chennai is in Tamil and may require subtitles for non-native speakers. The film contains graphic violence and mature themes. |
| **Gaayam (1993)** | Directed by Ram Gopal Varma, Gaayam is a crime drama that delves into the world of gangsters and the police, offering a gritty look at the criminal underworld and its consequences. The film is praised for its realistic portrayal and is considered a classic in the genre. | Gaayam is in Telugu, so viewers may need subtitles. The film's era and pacing might differ from more contemporary viewers' expectations. |
| **D (2005)** | This film is a crime drama that explores the underworld, focusing on a young man's rise in the criminal world. It offers a gritty and intense look at the life of gangsters, police encounters, and the violence that ensues. With its focus on character development, D provides a deep dive into the motivations and consequences of criminal life. | D is in Hindi, making it more accessible to a broader Indian audience. However, it contains mature themes and violence. |
| **Mardaani (2014)** | Mardaani stands out for its focus on a female police officer's battle against the criminal underworld, particularly in the context of human trafficking. It offers a unique perspective on police encounters and the fight against crime, with a strong, gritty narrative. | The film is in Hindi and deals with very mature and disturbing themes, including child trafficking and abuse, so viewer discretion is strongly advised. |

Each of these films offers a unique perspective on the Indian gangster epic genre, exploring themes of crime, violence, and the struggle between law enforcement and the criminal underworld. They are recommended for viewers looking for gritty, realistic portrayals of these themes, with the understanding that some may contain mature content or require subtitles for full appreciation.

## 7. Technical Architecture Deep-Dive

In [7]:
render_mermaid('''
graph TD
    subgraph Corpus ["📦 Corpus Pipeline (95,090 films)"]
        A["8 Raw Parquet Layers\n347k base movies"] --> B["Data Curation\nDedup, score norm, genre filter"]
        B --> C["Embedding Text Template\nTitle + Year + Genre + Plot"]
        C --> D["BGE-base Embedding\n768-d float32 vectors"]
        D --> E["Qdrant Cloud\nCOSINE distance, 14 payload indices"]
    end
    subgraph Quality ["🛡 Quality Guards"]
        F["Global Guard: year>0, votes>0"]
        G["HiddenGem: 100≤votes≤10k, rating≥7.0"]
        H["Gem Score Blend: 60% gem + 40% semantic"]
    end
    subgraph Eval ["📊 Evaluation (21 queries)"]
        I["6 Retrieval Strategies"] --> J["Mode A: Dense"]
        J --> K["Mode B: +Reranker"]
        K --> L["Mode C: +LLM"]
    end
    E ~~~ F
    H ~~~ I
''')


---
## 🙏 Acknowledgements
- **IMDB Non-Commercial Dataset** — base film metadata
- **Groq API** — `llama-3.3-70b-versatile` inference
- **Qdrant Cloud** — managed vector database
- **BAAI** — `bge-base-en-v1.5` and `bge-reranker-large` models

If you found this notebook useful, please **upvote** ⬆️ and feel free to fork it!

> 💬 Questions? Drop a comment below or open an issue on GitHub.
